In [ ]:
import pandas as pd
import yfinance as yf

In [ ]:
start_date ='2015-01-01'
end_date = '2025-12-31'
final_pd = pd.read_csv('CSV/TICKERS.csv', keep_default_na=False) # reading CSV
final_pd['EQUITY'] = final_pd['EQUITY'].str.replace('.','-',regex=False)
#print(final_pd)

In [ ]:
#EQUITY
equity_tickers = []
for t in final_pd['EQUITY']:
    clean_equity = t.strip() #убирает пробелы 
    if clean_equity != '':
        equity_tickers.append(clean_equity)
if len(equity_tickers) > 0:
    equity_data = yf.download(equity_tickers, start=start_date, end=end_date)['Close']
    equity_data.index = pd.to_datetime(equity_data.index).date
    equity_data.to_excel('DATA/EQUITY.xlsx')

In [ ]:
#CRYPTO
crypto_tickers = []
for t in final_pd['CRYPTO']:
    clean_crypto = t.strip()
    if clean_crypto !='':
        crypto_tickers.append(clean_crypto)
if len(crypto_tickers) > 0:
    crypto_data = yf.download(crypto_tickers, start=start_date, end=end_date)['Close']
    crypto_data.index = pd.to_datetime(crypto_data.index).date
    crypto_data.to_excel('DATA/CRYPTO.xlsx')

In [ ]:
#FOREX
forex_tickers = []
for t in final_pd['FOREX']:
    forex_clean = t.strip()
    if forex_clean != '':
        forex_tickers.append(forex_clean)
if len(forex_tickers) > 0:
    forex_data = yf.download(forex_tickers,start=start_date, end=end_date)['Close']
    forex_data.index = pd.to_datetime(forex_data.index).date
    forex_data.to_excel('DATA/FOREX.xlsx')
print(forex_data)

In [ ]:
#COMODITIES
comodities_ticker = []
for t in final_pd['COMODITIES']:
    comodities_clean = t.strip()
    if comodities_clean != '':
        comodities_ticker.append(comodities_clean)
if len(comodities_ticker) > 0:
    comodities_data = yf.download(comodities_ticker,start=start_date, end=end_date)['Close']
    comodities_data.index = pd.to_datetime(comodities_data.index).date
    comodities_data.to_excel('DATA/COMODITIES.xlsx')
print(comodities_data)

In [43]:
# Calendar Alignment / 24/7 Grid
# Forward_Fill
master_df = None
files_to_merge =['EQUITY.xlsx',
                 'FOREX.xlsx',
                 'CRYPTO.xlsx',
                 'COMODITIES.xlsx']
for file_name in files_to_merge:
    files_path = 'DATA/'+ file_name
    try:
        df = pd.read_excel(files_path)
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = [col[0] if not str(col[0]).startswith('Unnamed') else col[1] for col in df.columns]
        df.rename(columns={df.columns[0]:'Date'}, inplace=True)
        df['Date'] = pd.to_datetime(df['Date']).dt.date
        df = df.dropna(subset=['Date'])
        df.set_index('Date', inplace=True)
        df = df[~df.index.duplicated(keep='first')]
        if master_df is None:
            master_df = df
        else:
            master_df = master_df.join(df, how='outer')
    except FileNotFoundError:
        print(file_name,'does not exist')
    except Exception as e:
        print(file_name,'editing error')
master_df = master_df.ffill().sort_index()
master_df.to_excel('DATA/FULL_Forward_Fill.xlsx')

In [42]:
# Trading/Business Calendar Alignment
base_file = 'DATA/EQUITY.xlsx'
try:
    df_base = pd.read_excel(base_file)
    df_base.rename(columns={df_base.columns[0]: 'Date'}, inplace=True)
    df_base['Date'] = pd.to_datetime(df_base['Date']).dt.date
    df_base = df_base.dropna(subset=['Date'])
    df_base.set_index('Date', inplace=True)
    df_base = df_base[~df_base.index.duplicated(keep='first')]
    master_df = df_base
except Exception as e:
    print('ERROR 1')
  
other_files = ['FOREX.xlsx',
                'CRYPTO.xlsx',
                'COMODITIES.xlsx']
if master_df is not None:
    for file_name in other_files:
        file_path = 'DATA/' + file_name
        try:
            df = pd.read_excel(file_path)
            if isinstance(df.columns, pd.MultiIndex):
                df.columns = [col[0] if not str(col[0]).startswith('Unnamed') else col[1] for col in df.columns]
            df.rename(columns={df.columns[0]: 'Date'}, inplace=True)
            df['Date'] = pd.to_datetime(df['Date']).dt.date
            df = df.dropna(subset=['Date'])
            df.set_index('Date', inplace=True)
            df = df[~df.index.duplicated(keep='first')]
            master_df = master_df.join(df, how='left')
        except FileNotFoundError:
            print(f'ERROR 2')
        except Exception as e:
            print('ERROR 3')
    master_df = master_df.ffill()
    master_df = master_df.sort_index()
    master_df.to_excel('DATA/FULL_Trading_Calendar.xlsx')
else: 
    print('ERROR 4')